# Kafka

In [ ]:
from confluent_kafka import Producer

conf = {
    "bootstrap.servers": "localhost:9092"
}
producer = Producer(conf)
producer.produce("my_topic", value="hello world")
producer.flush()


In [ ]:
from confluent_kafka import Consumer

# Config
conf = {
    'bootstrap.servers': 'localhost:9092',
    'group.id': 'test-group',
    'auto.offset.reset': 'earliest'  # start from beginning if no offset
}

consumer = Consumer(conf)
consumer.subscribe(['weather-raw'])

print("Consuming...")
while True:
    msg = consumer.poll(1.0)
    if msg is None:
        continue
    if msg.error():
        print("Error:", msg.error())
        continue
    value = msg.value().decode('utf-8')
    print(f"type: {type(value)} - {value}")

consumer.close()


# spark stream

In [23]:
%%sh
wget -O ./jars/spark-sql-kafka-0-10_2.13-3.2.4.jar https://repo1.maven.org/maven2/org/apache/spark/spark-sql-kafka-0-10_2.13/3.2.4/spark-sql-kafka-0-10_2.13-3.2.4.jar

--2025-05-03 13:20:09--  https://repo1.maven.org/maven2/org/apache/spark/spark-sql-kafka-0-10_2.13/3.2.4/spark-sql-kafka-0-10_2.13-3.2.4.jar
Resolving repo1.maven.org (repo1.maven.org)... 146.75.40.209, 151.101.196.209, 2a04:4e42:a::209, ...
Connecting to repo1.maven.org (repo1.maven.org)|146.75.40.209|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 426685 (417K) [application/java-archive]
Saving to: ‘./jars/spark-sql-kafka-0-10_2.13-3.2.4.jar’

     0K .......... .......... .......... .......... .......... 11%  193K 2s
    50K .......... .......... .......... .......... .......... 23%  177K 2s
   100K .......... .......... .......... .......... .......... 35% 13.2M 1s
   150K .......... .......... .......... .......... .......... 47% 32.9M 1s
   200K .......... .......... .......... .......... .......... 59%  196K 1s
   250K .......... .......... .......... .......... .......... 71%  613K 0s
   300K .......... .......... .......... .......... .......... 83%  

In [7]:
from pyspark.sql import SparkSession
from pyspark import SparkConf
from contextlib import contextmanager
from pyspark.sql.types import StructType, StringType, IntegerType, TimestampType
from pyspark.sql.functions import col, regexp_replace, trim, regexp_extract, to_timestamp, from_json
from pyspark.sql.types import DoubleType

@contextmanager
def SparkIO(app_name: str = "test_app"):    
    packages = [
        # 'org.apache.kafka:kafka-clients:3.2.1',
        # "org.apache.spark:spark-sql-kafka-0-10_2.12:3.2.4",
        "jars/spark-sql-kafka-0-10_2.13-3.2.4.jar"
        # "jars/spark-sql-kafka-0-10_2.13-3.2.4.jar
    ]

            # .config("spark.jars.packages", ",".join(packages))\
    spark = SparkSession.builder\
            .master('local[*]')\
            .config('spark.app.name', f'{app_name}')\
            .config("spark.jars", ",".join(packages))\
            .getOrCreate()

    spark.sparkContext.setLogLevel("WARN")
    print(f'Created SparkSession app {app_name}')
    try:
        yield spark
    except Exception as e:
        print(f'Error in SparkSession app {app_name}: {e}')
        raise
    finally:
        print(f'Stop SparkSession app {app_name}')
        spark.stop()

if __name__ == "__main__":
    with SparkIO("test_app") as spark:
        # Read from Kafka
        df = spark.readStream\
            .format("kafka")\
            .option("kafka.bootstrap.servers", "kafka-broker-1:9094")\
            .option("subscribe", "weather-raw")\
            .load()

        df = df.selectExpr("CAST(value AS STRING)")
        schema = StructType() \
            .add("current_time", StringType()) \
            .add("status", StringType()) \
            .add("temp_c", StringType()) \
            .add("realfeel®", StringType()) \
            .add("wind", StringType()) \
            .add("wind gusts", StringType()) \
            .add("humidity", StringType()) \
            .add("indoor humidity", StringType()) \
            .add("dew point", StringType()) \
            .add("pressure", StringType()) \
            .add("cloud cover", StringType()) \
            .add("visibility", StringType()) \
            .add("cloud ceiling", StringType()) \
            .add("timestamp", StringType())
        
        parsed_df = df.select(from_json(col("value"), schema).alias("data")).select("data.*")

        # Clean and standardize
        cleaned_df = parsed_df \
            .withColumn("temp_c", regexp_extract(col("temp_c"), r"([\d.]+)", 1).cast(DoubleType())) \
            .withColumn("realfeel", regexp_extract(col("realfeel®"), r"([\d.]+)", 1).cast(DoubleType())) \
            .withColumn("wind_kmh", regexp_extract(col("wind"), r"(\d+)", 1).cast(DoubleType())) \
            .withColumn("wind_gusts_kmh", regexp_extract(col("wind gusts"), r"(\d+)", 1).cast(DoubleType())) \
            .withColumn("humidity", regexp_replace(col("humidity"), "%", "").cast(DoubleType())) \
            .withColumn("indoor_humidity", regexp_extract(col("indoor humidity"), r"(\d+)", 1).cast(DoubleType())) \
            .withColumn("dew_point_c", regexp_extract(col("dew point"), r"([\d.]+)", 1).cast(DoubleType())) \
            .withColumn("pressure_mb", regexp_extract(col("pressure"), r"(\d+)", 1).cast(DoubleType())) \
            .withColumn("cloud_cover", regexp_replace(col("cloud cover"), "%", "").cast(DoubleType())) \
            .withColumn("visibility_km", regexp_extract(col("visibility"), r"([\d.]+)", 1).cast(DoubleType())) \
            .withColumn("cloud_ceiling_m", regexp_extract(col("cloud ceiling"), r"(\d+)", 1).cast(DoubleType())) \
            .withColumn("timestamp", to_timestamp("timestamp"))  # Druid likes proper timestamps

        # Optional: drop original columns with odd names or keep selected ones
        final_df = cleaned_df.select(
            "timestamp",
            "temp_c",
            "realfeel",
            "wind_kmh",
            "wind_gusts_kmh",
            "humidity",
            "indoor_humidity",
            "dew_point_c",
            "pressure_mb",
            "cloud_cover",
            "visibility_km",
            "cloud_ceiling_m"
        )

        # Write to console
        query = final_df.writeStream\
            .outputMode("append")\
            .format("console")\
            .option("truncate", False)\
            .start()
        query.awaitTermination()

Created SparkSession app test_app


25/05/10 12:12:12 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-8fee64b1-0bba-437d-99f7-2291a1ff384f. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/05/10 12:12:12 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+---------+------+--------+--------+--------------+--------+---------------+-----------+-----------+-----------+-------------+---------------+
|timestamp|temp_c|realfeel|wind_kmh|wind_gusts_kmh|humidity|indoor_humidity|dew_point_c|pressure_mb|cloud_cover|visibility_km|cloud_ceiling_m|
+---------+------+--------+--------+--------------+--------+---------------+-----------+-----------+-----------+-------------+---------------+
+---------+------+--------+--------+--------------+--------+---------------+-----------+-----------+-----------+-------------+---------------+



-------------------------------------------
Batch: 1
-------------------------------------------
+-------------------+------+--------+--------+--------------+--------+---------------+-----------+-----------+-----------+-------------+---------------+
|timestamp          |temp_c|realfeel|wind_kmh|wind_gusts_kmh|humidity|indoor_humidity|dew_point_c|pressure_mb|cloud_cover|visibility_km|cloud_ceiling_m|
+-------------------+------+--------+--------+--------------+--------+---------------+-----------+-----------+-----------+-------------+---------------+
|2025-05-10 19:13:04|27.0  |31.0    |11.0    |11.0          |83.0    |83.0           |24.0       |1007.0     |31.0       |8.0          |12200.0        |
+-------------------+------+--------+--------+--------------+--------+---------------+-----------+-----------+-----------+-------------+---------------+



-------------------------------------------
Batch: 2
-------------------------------------------
+-------------------+------+--------+--------+--------------+--------+---------------+-----------+-----------+-----------+-------------+---------------+
|timestamp          |temp_c|realfeel|wind_kmh|wind_gusts_kmh|humidity|indoor_humidity|dew_point_c|pressure_mb|cloud_cover|visibility_km|cloud_ceiling_m|
+-------------------+------+--------+--------+--------------+--------+---------------+-----------+-----------+-----------+-------------+---------------+
|2025-05-10 19:15:04|27.0  |31.0    |11.0    |11.0          |83.0    |83.0           |24.0       |1007.0     |31.0       |8.0          |12200.0        |
+-------------------+------+--------+--------+--------------+--------+---------------+-----------+-----------+-----------+-------------+---------------+



...

-------------------------------------------
Batch: 4
-------------------------------------------
+-------------------+------+--------+--------+--------------+--------+---------------+-----------+-----------+-----------+-------------+---------------+
|timestamp          |temp_c|realfeel|wind_kmh|wind_gusts_kmh|humidity|indoor_humidity|dew_point_c|pressure_mb|cloud_cover|visibility_km|cloud_ceiling_m|
+-------------------+------+--------+--------+--------------+--------+---------------+-----------+-----------+-----------+-------------+---------------+
|2025-05-10 19:18:06|27.0  |32.0    |9.0     |9.0           |83.0    |83.0           |24.0       |1007.0     |76.0       |16.0         |12200.0        |
+-------------------+------+--------+--------+--------------+--------+---------------+-----------+-----------+-----------+-------------+---------------+



ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/opt/conda/lib/python3.9/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
RuntimeError: reentrant call inside <_io.BufferedReader name=62>

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/conda/lib/python3.9/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/conda/lib/python3.9/site-packages/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving
ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/opt/conda/lib/python3.9/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/opt/conda/lib/python3.9/soc

Error in SparkSession app test_app: An error occurred while calling o814.awaitTermination
Stop SparkSession app test_app


Py4JError: An error occurred while calling o814.awaitTermination

25/05/10 12:19:06 ERROR MicroBatchExecution: Query [id = ca5fefcc-074e-4d19-9904-58ff26d1b2c7, runId = f0b4c093-d8cd-45bc-9ba7-541dea138cd3] terminated with error
java.lang.NullPointerException
	at org.apache.spark.sql.kafka010.KafkaOffsetReaderConsumer.getSortedExecutorList(KafkaOffsetReaderConsumer.scala:484)
	at org.apache.spark.sql.kafka010.KafkaOffsetReaderConsumer.getOffsetRangesFromResolvedOffsets(KafkaOffsetReaderConsumer.scala:539)
	at org.apache.spark.sql.kafka010.KafkaMicroBatchStream.planInputPartitions(KafkaMicroBatchStream.scala:191)
	at org.apache.spark.sql.execution.datasources.v2.MicroBatchScanExec.partitions$lzycompute(MicroBatchScanExec.scala:44)
	at org.apache.spark.sql.execution.datasources.v2.MicroBatchScanExec.partitions(MicroBatchScanExec.scala:44)
	at org.apache.spark.sql.execution.datasources.v2.DataSourceV2ScanExecBase.supportsColumnar(DataSourceV2ScanExecBase.scala:93)
	at org.apache.spark.sql.execution.datasources.v2.DataSourceV2ScanExecBase.supportsColumna

In [16]:
from pyspark.sql.functions import from_json, col
packages = [
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0",
    'org.apache.kafka:kafka-clients:3.2.4'
]
spark = SparkSession.builder\
         .appName("KafkaSparkStreaming")\
         .config("spark.jars.packages", ",".join(packages))\
         .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# schema = StructType()\
#     .add("current_time", StringType(), True)\
#     .add("status", StringType(), True)\
#     .add("temp_c", StringType(), True)\
#     .add("realfeel®", StringType(), True)\
#     .add("realfeel shade™", StringType(), True)\
#     .add("max uv index", StringType(), True)\
#     .add("wind", StringType(), True)\
#     .add("wind gusts", StringType(), True)\
#     .add("humidity", StringType(), True)\
#     .add("indoor humidity", StringType(), True)\
#     .add("dew point", StringType(), True)\
#     .add("pressure", StringType(), True)\
#     .add("cloud cover", StringType(), True)\
#     .add("visibility", StringType(), True)\
#     .add("cloud ceiling", StringType(), True)

df = spark.readStream\
    .format("kafka")\
    .option("kafka.bootstrap.servers", "localhost:9092")\
    .option("subscribe", "weather-raw")\
    .option("startingOffsets", "earliest")\
    .load()

# df_parsed = df.selectExpr("CAST(value AS STRING) as json_str") \
#     .select(from_json(col("json_str"), schema).alias("data")) \
#     .select("data.*")
# # df_filtered = df_parsed.filter(col("event") == "click")

# query = df_parsed.writeStream \
#     .outputMode("append") \
#     .format("console") \
#     .option("truncate", False) \
#     .start()

# query.awaitTermination()
# # {'current_time': '10:52 AM', 
# #  'status': 'Mostly sunny', 
# #  'temp_c': '33°C\n', 
# #  'realfeel®': '38°', 
# #  'realfeel shade™': '36°', 
# #  'max uv index': '5 Moderate', 
# #  'wind': 'ESE 20 km/h', 
# #  'wind gusts': '20 km/h', 
# #  'humidity': '58%', 
# #  'indoor humidity': '58% (Extremely Humid)', 
# #  'dew point': '24° C', 
# #  'pressure': '↔ 1010 mb', 'cloud cover': '30%', 'visibility': '16 km', 'cloud ceiling': '600 m'}
spark.stop()

AnalysisException:  Failed to find data source: kafka. Please deploy the application as per the deployment section of "Structured Streaming + Kafka Integration Guide".        

In [11]:
!pip show pyspark

Name: pyspark
Version: 3.2.4
Summary: Apache Spark Python API
Home-page: https://github.com/apache/spark/tree/master/python
Author: Spark Developers
Author-email: dev@spark.apache.org
License: http://www.apache.org/licenses/LICENSE-2.0
Location: /opt/conda/lib/python3.9/site-packages
Requires: py4j
Required-by: 


In [9]:
spark.stop()